# Single-organoid graph build + crypt segmentation (debug notebook)
This notebook builds the **cell adjacency graph** for **one organoid** from the mesh + nuclei CSV, runs **crypt segmentation**, and produces a **minimal Plotly visualization** (markers + crypt overlays).

The goal is to make it easy to tweak hyperparameters by editing values in-place and inspecting intermediate variables.


In [88]:
# --- Imports ---
import os
import glob
import numpy as np
import pandas as pd

# Pipeline modules (from the scripts you provided)
from pathlib import Path
import sys

# notebooks/ is sibling of scripts/
ROOT = Path.cwd().resolve()
SCRIPTS_DIR = (ROOT / ".." / "scripts").resolve()

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import run_graph_preprocess as rgp
import run_crypt_segmentation as rcs

from organograph.mesh.OrganoidMesh import OrganoidMesh
from organograph.graph.build import build_organoid_graph, add_vertex_field_to_graph
from organograph.io_utils.cells_table import prepare_cells_table, make_nuclei_extractor
from organograph.mesh.hks import compute_hks
from organograph.crypts.vocab import compute_vocabulary_encoding

from organograph.mesh.transform import ensure_mesh_graph_aligned
from organograph.crypts.segment import segment_crypts_organoid

# Plotting utilities (attached)
from organograph.plotting.graphs import plot_graph_by_markers, add_region_overlays, plot_organoid_graph
from organograph.plotting.meshes import plot_organoid_mesh

## 1) Dataset paths + selection
Edit these to point at your dataset. The defaults mirror `run_graph_preprocess.py`.

In [89]:
# -----------------------
# CONFIG (edit these)
# -----------------------

# You can keep these as the defaults from run_graph_preprocess.py, or override them here.
MESH_DATA_DIR = getattr(rgp, "MESH_DATA_DIR", None)
CELLS_CSV     = getattr(rgp, "CELLS_CSV", None)
VOCAB_PATH    = "../sim/vocab.npz"

# Which timepoint to pull a single organoid from (used only for mesh discovery convenience)
TIMEPOINT = (getattr(rgp, "timepoints", None) or ["day4p5"])[0]

# If you already know the exact mesh file, set it here and skip discovery, otherwise set None
WELL   = "B03"
ORG_ID = "144"

# WELL   = "B02"
# ORG_ID = "124"

# WELL   = "B04"
# ORG_ID = "4"

# WELL   = "B05"
# ORG_ID = "54"

# WELL   = "B02"
# ORG_ID = "115"

# WELL   = "B02"
# ORG_ID = "115"


MESH_PATH = (
    Path(MESH_DATA_DIR)
    / TIMEPOINT
    / "251130R0.zarr"
    / WELL[0]           # "B"
    / WELL[1:]          # "03"
    / "2_zillum_registered"
    / "meshes"
    / "nnorg_corrected_annotated_by_projection"
    / f"{ORG_ID}.vtp"
)

print("MESH_DATA_DIR =", MESH_DATA_DIR)
print("CELLS_CSV     =", CELLS_CSV)
print("VOCAB_PATH    =", VOCAB_PATH)
print("TIMEPOINT     =", TIMEPOINT)
print("MESH_PATH     =", MESH_PATH)

MESH_DATA_DIR = /home/fmoller/Projects/LearningOrganoids/organograph/../NicoleData/20251201/fractal_output
CELLS_CSV     = /home/fmoller/Projects/LearningOrganoids/organograph/../NicoleData/20251201/cell_features_class.csv
VOCAB_PATH    = ../sim/vocab.npz
TIMEPOINT     = day4p5
MESH_PATH     = /home/fmoller/Projects/LearningOrganoids/organograph/../NicoleData/20251201/fractal_output/day4p5/251130R0.zarr/B/03/2_zillum_registered/meshes/nnorg_corrected_annotated_by_projection/144.vtp


## 2) Pick one organoid mesh
If `MESH_PATH=None`, we auto-discover meshes using the same config logic as the preprocessing script and pick the first one.

In [90]:
# --- Auto-discover a single mesh (if needed) ---
if MESH_PATH is None:
    # Use the same restrictive config from run_graph_preprocess.py
    zarr_names = getattr(rgp, "zarr_names", None)
    rounds     = getattr(rgp, "rounds", None)
    meshes     = getattr(rgp, "meshes", None)
    wells      = getattr(rgp, "wells", None)

    mesh_paths = rgp.discover_mesh_paths(
        MESH_DATA_DIR,
        timepoints=[TIMEPOINT],
        zarr_names=zarr_names,
        rounds=rounds,
        meshes=meshes,
        wells=wells,
    )
    mesh_paths = sorted(mesh_paths)

    print(f"Discovered {len(mesh_paths)} meshes for timepoint={TIMEPOINT}")
    for p in mesh_paths[:10]:
        print("  ", p)

    if not mesh_paths:
        raise FileNotFoundError("No meshes found. Set MESH_PATH manually or check MESH_DATA_DIR / config.")

    MESH_PATH = mesh_paths[0]
    print("\nAuto-picked MESH_PATH =", MESH_PATH)

# Parse identifiers from mesh path (label_uid is the organoid id used everywhere downstream)
rec = rgp.parse_mesh_path(MESH_PATH)
label_uid = rec.get("label_uid", None)
timepoint = rec.get("timepoint", None)

print("label_uid =", label_uid)
print("timepoint =", timepoint)

label_uid = day4p5_B03_144
timepoint = day4p5


## 3) Build the cell graph (always from scratch)
This is the Stage-1 pipeline: load mesh → look up nuclei rows in CSV → build adjacency graph → attach HKS + vocab encoding.

All intermediate variables are left accessible for debugging (e.g. `cells_df`, `mesh`, `G`, `aux`, `hks`, `encoding`).

In [91]:
# --- Load cells table and build nuclei extractor ---
if not os.path.exists(CELLS_CSV):
    raise FileNotFoundError(f"CELLS_CSV not found: {CELLS_CSV}")

cells_df = pd.read_csv(CELLS_CSV)
cells_df = prepare_cells_table(cells_df, label_col="label_uid")

# COORD_COLS / MARKER_COLS / marker_postprocess come from your preprocessing script
COORD_COLS = getattr(rgp, "COORD_COLS", None)
MARKER_COLS = getattr(rgp, "MARKER_COLS", None)
marker_postprocess = getattr(rgp, "marker_postprocess", None)

extractor = make_nuclei_extractor(
    cells_df,
    label_col="label_uid",
    xyz_cols=COORD_COLS,
    marker_cols=MARKER_COLS,
    marker_postprocess_fn=marker_postprocess,
)

# --- Load vocab ---
if not os.path.exists(VOCAB_PATH):
    raise FileNotFoundError(f"VOCAB_PATH not found: {VOCAB_PATH}")

vocab = np.load(VOCAB_PATH, allow_pickle=True)

# Optional: validate like the script does (if present)
validate_vocab_npz = getattr(rgp, "validate_vocab_npz", None)
if validate_vocab_npz is not None:
    validate_vocab_npz(vocab)

# --- Load + normalize mesh ---
mesh = OrganoidMesh(str(MESH_PATH))
mesh.normalize_inplace()
mesh.label_uid = label_uid

# --- Build graph ---
G, aux = build_organoid_graph(mesh=mesh, extract_fn=extractor)
G.graph["label_uid"] = label_uid
G.graph["timepoint"] = timepoint
G.graph["mesh_path"] = MESH_PATH

print("Graph built.")
print("  N_cells =", G.number_of_nodes())
print("  N_edges =", G.number_of_edges())
print("  marker_names =", G.graph.get("marker_names", None))

/home/fmoller/Projects/LearningOrganoids/organograph/src/organograph/io_utils/cells_table.py:121: RuntimeWarning:

prepare_cells_table: 'label_uid' index contains duplicates. This is usually expected (multiple cells per organoid). df.loc[label_uid] will return multiple rows.



Duplicate projections: 6 vertices with duplicates, 6 duplicate cell pairs.
Graph built.
  N_cells = 1301
  N_edges = 3897
  marker_names = ['0.C02.percentile99_class', '0.C03.percentile99_class', '0.C04.percentile99_class', '1.C02.percentile99_class', '1.C03.percentile99_class', '1.C04.percentile99_class', '2.C04.percentile99_class']


## 4) Attach HKS + vocab encoding
These are required/expected downstream. If something fails here, it usually indicates a mesh issue (e.g. eigen-decomp/HKS) or a vocab mismatch.

In [92]:
# --- HKS (required) ---
HKS_TIMES = getattr(rgp, "HKS_TIMES", None)
if HKS_TIMES is None:
    raise ValueError("run_graph_preprocess.py does not define HKS_TIMES (unexpected).")

mesh._eig_decomp()  # Laplace eigenpairs (required for HKS)
hks = compute_hks(mesh, t=HKS_TIMES, coeffs=False)  # (V, T)
add_vertex_field_to_graph(G, hks, "hks")
G.graph["hks_times"] = np.asarray(HKS_TIMES, float)

print("Attached HKS:", "hks" in next(iter(G.nodes(data=True)))[1])

# --- vocab encoding ---
encoding, _, _, _ = compute_vocabulary_encoding(vocab, mesh)
add_vertex_field_to_graph(G, encoding, "vocab_encoding")
G.graph["vocab_path"] = VOCAB_PATH

print("Attached vocab_encoding:", "vocab_encoding" in next(iter(G.nodes(data=True)))[1])

Attached HKS: True
Attached vocab_encoding: True


## 5) Crypt segmentation (single organoid)
Tune hyperparameters by editing `SEGMENT_KWARGS` below.

In [ ]:
# -----------------------
# SEGMENTATION CONFIG (edit these)
# -----------------------

S = 1.5
n_bins = 80
bin_edges   = np.linspace(0.0, S, n_bins + 1)
BIN_CENTERS = 0.5 * (bin_edges[:-1] + bin_edges[1:])

CRYPT_VOCAB_IDX  = list(getattr(rcs, "CRYPT_VOCAB_IDX", []))
NECK_VOCAB_IDX   = getattr(rcs, "NECK_VOCAB_IDX", None)
DEBUG            = True

# Main hyperparameters live here (same dict name as in the script).
# Add/remove knobs as needed (examples are in run_crypt_segmentation.py).
SEGMENT_KWARGS = {
    # "neck_seed_thresh": 0.3,
    # "far_scan_w_neck": 10,
}

print("BIN_CENTERS.shape =", getattr(BIN_CENTERS, "shape", None))
print("CRYPT_VOCAB_IDX   =", CRYPT_VOCAB_IDX)
print("NECK_VOCAB_IDX    =", NECK_VOCAB_IDX)
print("DEBUG             =", DEBUG)
print("SEGMENT_KWARGS    =", SEGMENT_KWARGS)

BIN_CENTERS.shape = (80,)
CRYPT_VOCAB_IDX   = [1, 2, 7]
NECK_VOCAB_IDX    = [0, 3, 4]
DEBUG             = True
SEGMENT_KWARGS    = {'neck_seed_thresh': 0.3, 'far_scan_w_neck': 10}


In [94]:
# --- Ensure mesh/graph alignment (important!) ---
ensure_mesh_graph_aligned(mesh, G)

# --- Run segmentation ---
out = segment_crypts_organoid(
    G=G,
    mesh=mesh,
    bin_centers=BIN_CENTERS,
    crypt_vocab_idx=CRYPT_VOCAB_IDX,
    neck_vocab_idx=None if NECK_VOCAB_IDX is None else list(NECK_VOCAB_IDX),
    debug=DEBUG,
    **(SEGMENT_KWARGS or {}),
)

if DEBUG:
    crypts, villi, d_norm, L_crypt, Circ, dbg = out
else:
    crypts, villi, d_norm, L_crypt, Circ = out
    dbg = None

print("Segmentation done.")
print("  n_crypts =", len(crypts) if crypts is not None else None)
print("  n_villi  =", len(villi) if villi is not None else None)

Segmentation done.
  n_crypts = 4
  n_villi  = 1


## 6) Minimal plotting (Plotly)
Plots the graph with marker positivity highlighted for **LGR5**, **AldoB**, **Sero**, **Lyzo**, and overlays detected crypt regions.

If your marker names differ, update `MARKER_MAP` below to match `G.graph['marker_names']`.

In [95]:
# -----------------------
# PLOTTING CONFIG (edit these)
# -----------------------

BACKEND = "plotly"
VIEW = dict(azim=-135, elev=25, projection="orthographic")

NODE_SIZE = 5
EDGE_WIDTH = 1.0

# Crypt overlay style
CRYPT_OVERLAY_COLOR = None           # set to e.g. "#FFD700" to force single-color
CRYPT_OVERLAY_COLORSCALE = "Turbo"
CRYPT_OVERLAY_ALPHA = 0.6
CRYPT_OVERLAY_SIZE_MULT = 1.7

# Marker definitions (use marker *names* here; graphs.py resolves them)
MARKER_MAP = [
    {"marker": 0,  "color": "#1f77b4", "name": "stem cell (LGR5+)"},
    {"marker": 2, "color": "#2ca02c", "name": "absorptive (AldoB+)"},
    {"marker": 3,  "color": "#d62728", "name": "enterochromaffin (Sero+)"},
    {"marker": 4,  "color": "#9467bd", "name": "paneth (Lyzo+)"},
]

# --- Base plot: markers ---
fig = plot_graph_by_markers(
    G,
    MARKER_MAP,
    backend=BACKEND,
    view=VIEW,
    node_size=NODE_SIZE,
    edge_width=EDGE_WIDTH,
)

# --- Overlay: one region per crypt (so each crypt can get its own color) ---
regions = [set(map(int, c)) for c in (crypts or [])]  # crypts is typically list[set[int]] already

# Decide whether to use single-color or colormap
overlay_colors = [CRYPT_OVERLAY_COLOR] if CRYPT_OVERLAY_COLOR else None
overlay_colorscale = None if CRYPT_OVERLAY_COLOR else CRYPT_OVERLAY_COLORSCALE

if regions:
    add_region_overlays(
        fig, G, regions,
        backend="plotly",
        colors=overlay_colors,
        colorscale=overlay_colorscale,
        size=NODE_SIZE * CRYPT_OVERLAY_SIZE_MULT,
        name_prefix="crypt",
        alpha=CRYPT_OVERLAY_ALPHA,
    )

fig

In [96]:
H = hks[:, 0] - np.mean(hks[:, 0])
fig = plot_organoid_mesh(
    mesh,
    vertex_values=H,
    backend="plotly",
    colorscale="RdBu_r",
    center_at_zero=True,
    view=dict(azim=-135, elev=25, projection="orthographic"),
)
fig.show()

In [97]:
fig = plot_organoid_graph(
    G,
    node_values=None,
    backend=BACKEND,
    view=VIEW,
    node_size=NODE_SIZE,
    edge_width=EDGE_WIDTH,
)

# --- Overlay: one region per neck (so each neck can get its own color) ---
regions = [set(map(int, c)) for c in (dbg["necks_grow"] or [])]  # necks is typically list[set[int]] already

# Decide whether to use single-color or colormap
overlay_colors = [CRYPT_OVERLAY_COLOR] if CRYPT_OVERLAY_COLOR else None
overlay_colorscale = None if CRYPT_OVERLAY_COLOR else CRYPT_OVERLAY_COLORSCALE

if regions:
    add_region_overlays(
        fig, G, regions,
        backend="plotly",
        colors=overlay_colors,
        colorscale=overlay_colorscale,
        size=NODE_SIZE * CRYPT_OVERLAY_SIZE_MULT,
        name_prefix="neck",
        alpha=CRYPT_OVERLAY_ALPHA,
    )

fig